# Data audit: what we have and what is missing
Checks the stacked multi-city table (and per-city `meta.json` files if present) for missing cities, missing columns, NaN / constant columns, failed weather fetches and suspicious fallback values.

In [ ]:
import json, glob
import numpy
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 200)

# --- point this at your output folder (the --out used in build_dataset.py)
CANDIDATES = ["../data_multi", "../data/multi_city", "data_multi", "data/multi_city", "../data/smoke_test", "data/smoke_test"]
OUT = next((Path(p) for p in CANDIDATES if (Path(p) / "all_cities_training.csv.gz").exists()), None)
assert OUT is not None, f"all_cities_training.csv.gz not found in {CANDIDATES}; set OUT manually"
print("using", OUT.resolve())

EXPECTED_CITIES = ["Ahmedabad","Delhi","Jaipur","Pune","Nagpur","Hyderabad","Chennai","Mumbai","Bengaluru",
                   "Kolkata","Guwahati","Lucknow","Bhubaneswar","Chandigarh"]
df = pd.read_csv(OUT / "all_cities_training.csv.gz")
print(f"{len(df):,} rows x {df.shape[1]} columns, {df.memory_usage(deep=True).sum()/1e6:.0f} MB")

using C:\Hackathon\Heat_Map\data\smoke_test
8,100 rows x 39 columns, 4 MB


## 1. Cities: present vs expected

In [ ]:
have = df["city"].value_counts()
city_tbl = pd.DataFrame({"expected": True}, index=EXPECTED_CITIES)
city_tbl["rows"] = have.reindex(EXPECTED_CITIES).fillna(0).astype(int)
city_tbl["present"] = city_tbl["rows"] > 0
print("MISSING cities:", city_tbl.index[~city_tbl.present].tolist() or "none")
print("Unexpected cities:", sorted(set(have.index) - set(EXPECTED_CITIES)) or "none")
city_tbl

## 2. Columns grouped by source: present vs missing

In [ ]:
GROUPS = {
  "Landsat (LST, indices)": ["LST","NDVI","NDBI","MNDWI","ALBEDO","LST_std","n_obs"],
  "Terrain / land cover":   ["ELEV","LANDCOVER","SLOPE_deg"],
  "GHSL":                   ["GHSL_HEIGHT","GHSL_BUILT_FRAC","GHSL_POP"],
  "OSM (needs --osm)":      ["BLDG_FRAC","WATER_FRAC","GREEN_FRAC"],
  "Neighbourhood":          ["NDVI_nbr150","NDVI_nbr450","BUILT_nbr150","BUILT_nbr450","HEIGHT_nbr150","HEIGHT_nbr450","BLDG_nbr150","BLDG_nbr450"],
  "Distances":              ["DIST_WATER_m","DIST_GREEN_m"],
  "Weather (Open-Meteo)":   ["MET_T2M","MET_RH","MET_WIND","MET_SW","MET_RAIN7","MET_RAIN30","MET_T2M_SEASON_MAX"],
  "Labels":                 ["LST_z","SUHI","IS_URBAN","HOTSPOT"],
  "Meta":                   ["city","climate","lon","lat","x","y","block_id"],
}
rows = []
for g, cols in GROUPS.items():
    for c in cols:
        rows.append((g, c, c in df.columns, df[c].isna().mean() * 100 if c in df.columns else np.nan))
col_tbl = pd.DataFrame(rows, columns=["group","column","in_table","pct_nan_overall"])
print("Columns absent from the table:", col_tbl.loc[~col_tbl.in_table,"column"].tolist())
print("Columns in table but not in any group:", [c for c in df.columns if c not in sum(GROUPS.values(), [])])
col_tbl

## 3. Missing values: city x column (% NaN)
The table is written after dropping rows with NaN `LST`/`NDVI`, so those are 0 by construction. Weather columns are city-level constants, so a failed fetch shows up as 100% NaN for the whole city.

In [ ]:
num = df.select_dtypes("number").columns.drop(["x","y","lon","lat"], errors="ignore")
nan_pct = df[num].isna().groupby(df["city"]).mean().mul(100).round(1)
nan_pct = nan_pct.reindex([c for c in EXPECTED_CITIES if c in nan_pct.index])
fig, ax = plt.subplots(figsize=(max(10, len(num)*0.45), 0.45*len(nan_pct)+2))
im = ax.imshow(nan_pct.values, aspect="auto", cmap="Reds", vmin=0, vmax=100)
ax.set_xticks(range(len(num)), num, rotation=90); ax.set_yticks(range(len(nan_pct)), nan_pct.index)
for i in range(nan_pct.shape[0]):
    for j in range(nan_pct.shape[1]):
        v = nan_pct.values[i, j]
        if v > 0: ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=7)
plt.colorbar(im, label="% NaN"); ax.set_title("% missing by city and column"); plt.tight_layout(); plt.show()

print("Columns with any NaN:")
nan_pct.loc[:, (nan_pct > 0).any()]

## 4. Weather: which cities lost it?

In [ ]:
MET = [c for c in GROUPS["Weather (Open-Meteo)"] if c in df.columns]
met_nan = df[MET].isna().all(axis=1).groupby(df["city"]).mean()
weather = pd.DataFrame({"weather_ok": met_nan < 0.5, "pct_rows_without_weather": (met_nan*100).round(1)})
print("RE-RUN weather for:", weather.index[~weather.weather_ok].tolist() or "none")
weather.reindex([c for c in EXPECTED_CITIES if c in weather.index])

## 5. Constant, empty or fallback columns per city
A column with a single value in a city carries no signal (expected for weather, suspicious elsewhere). `DIST_WATER_m` / `DIST_GREEN_m` fall back to a fixed value when no water / green pixels are found.

In [ ]:
CONST_OK = set(MET) | {"n_obs"}
const = df.groupby("city")[list(num)].nunique(dropna=True)
const_flags = [(c, k) for c in const.index for k in const.columns if const.loc[c, k] <= 1 and k not in CONST_OK]
print("Constant / all-NaN columns (excluding weather & n_obs):")
if const_flags:
    display(pd.DataFrame(const_flags, columns=["city","column"]).sort_values(["city","column"]))
else:
    print("none")

fb = []
for k in ("DIST_WATER_m","DIST_GREEN_m"):
    if k in df:
        top = df.groupby("city")[k].agg(lambda s: s.value_counts(normalize=True).iloc[0]*100)
        fb.append(top.rename(f"{k}: % at modal value"))
print("Share of pixels at the single most common distance (high = fallback / no water found):")
pd.concat(fb, axis=1).round(1)

## 6. Coverage per city: rows, observations, area, zeros

In [ ]:
cov = df.groupby("city").agg(rows=("LST","size"), lst_mean=("LST","mean"), lst_min=("LST","min"), lst_max=("LST","max"),
                             n_obs_mean=("n_obs","mean"), n_obs_min=("n_obs","min"),
                             pct_hotspot=("HOTSPOT", lambda s: s.mean()*100),
                             pct_urban=("IS_URBAN", lambda s: s.mean()*100),
                             blocks=("block_id","nunique")).round(2)
for k in ("GHSL_BUILT_FRAC","GHSL_HEIGHT","GHSL_POP"):
    if k in df: cov[f"{k}_pct_zero"] = df[k].eq(0).groupby(df["city"]).mean().mul(100).round(1)
if "LANDCOVER" in df: cov["landcover_classes"] = df.groupby("city")["LANDCOVER"].nunique()
cov.reindex([c for c in EXPECTED_CITIES if c in cov.index])

## 7. Per-city metadata (scene counts, weather attrs) from `meta.json`

In [ ]:
metas = []
for f in sorted(glob.glob(str(OUT / "cities" / "*" / "meta.json"))):
    m = json.load(open(f)); m["_dir"] = Path(f).parent.name; metas.append(m)
if metas:
    meta = pd.DataFrame(metas).set_index("_dir")
    meta["has_weather"] = meta["MET_T2M"].notna() if "MET_T2M" in meta else False
    print("Cities with a table.csv.gz but no meta.json:",
          sorted(set(p.parent.name for p in (OUT/"cities").glob("*/table.csv.gz")) - set(meta.index)) or "none")
    display(meta[[c for c in ["city","n_scenes","hotspot_threshold_C","epsg","has_weather"] if c in meta]])
else:
    print("no meta.json files found under", OUT / "cities")

## 8. Issue summary

In [ ]:
issues = []
for c in city_tbl.index[~city_tbl.present]: issues.append((c, "city missing from table"))
for c in weather.index[~weather.weather_ok]: issues.append((c, "weather missing (Open-Meteo failed)"))
for c in col_tbl.loc[~col_tbl.in_table, "column"]: issues.append(("ALL", f"column absent: {c}"))
for c, k in const_flags: issues.append((c, f"column constant/empty: {k}"))
for c in nan_pct.index:
    for k in nan_pct.columns:
        if nan_pct.loc[c, k] > 0 and k not in MET: issues.append((c, f"{nan_pct.loc[c,k]:.0f}% NaN in {k}"))
issues = pd.DataFrame(issues, columns=["city","issue"])
print(f"{len(issues)} issue(s)")
issues